In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GATConv
from torch_scatter import scatter_sum

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EdgeFuzzyRuleLayer(nn.Module):
    """
    Edge-level fuzzy rules with patient-specific modulation.
    Different patients → different edge coefficients for the same protein.
    """
    def __init__(self, num_features=2, patient_emb_dim=64):
        super().__init__()
        # Learnable thresholds and sharpness (shared across edges)
        self.theta = nn.Parameter(torch.randn(num_features))
        self.alpha = nn.Parameter(torch.ones(num_features))
        
        # Patient-specific modulation network
        # Transforms patient embedding → personalized rule weights
        self.patient_modulator = nn.Sequential(
            nn.Linear(patient_emb_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_features),
            nn.Sigmoid()  # Weights in [0, 1]
        )
    
    def forward(self, protein_relevance, expression_value, patient_embeddings):
        """
        Args:
            protein_relevance: [num_edges] disease scores of proteins
            expression_value: [num_edges] expression values
            patient_embeddings: [num_edges, emb_dim] embeddings of SOURCE patients
            
        Returns:
            edge_coeffs: [num_edges] learned coefficients
            rule_activations: [num_edges, 2] base fuzzy activations
            patient_weights: [num_edges, 2] patient-specific modulation
        """
        # Stack edge features: [num_edges, 2]
        edge_features = torch.stack([protein_relevance, expression_value], dim=-1)
        
        # Base fuzzy rule activation (same for all patients):
        # r_i = sigmoid(alpha_i * (feature_i - theta_i))
        rule_activations = torch.sigmoid(
            self.alpha * (edge_features - self.theta)
        )  # [num_edges, 2]
        
        # Patient-specific modulation (different for each patient):
        patient_weights = self.patient_modulator(patient_embeddings)  # [num_edges, 2]
        
        # Combine: base rules × patient context
        edge_coeffs = (rule_activations * patient_weights).sum(dim=-1)  # [num_edges]
        
        # Final sigmoid to ensure [0, 1]
        edge_coeffs = torch.sigmoid(edge_coeffs)
        
        return edge_coeffs, rule_activations, patient_weights


class HeteroFireGNN_EdgeLevel(nn.Module):
    def __init__(self, data, hidden_channels, out_channels, heads=2, dropout_rate=0.5):
        super().__init__()
        
        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.patient_protein_edge_type = ('Patient', 'express', 'Protein')
        
        # Node embeddings
        self.embeddings = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        
        # GNN layers
        self.conv1 = HeteroConv({
            edge_type: GATConv((-1, -1), hidden_channels, heads, add_self_loops=False)
            for edge_type in data.edge_types
        })
        
        self.conv2 = HeteroConv({
            edge_type: GATConv(hidden_channels * heads, hidden_channels, heads=1, add_self_loops=False)
            for edge_type in data.edge_types
        })
        
        # Edge-level fuzzy rules with patient modulation
        self.edge_fuzzy_rules = EdgeFuzzyRuleLayer(
            num_features=2, 
            patient_emb_dim=hidden_channels
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_channels, out_channels)
        )
    
    def forward(self, edge_index_dict, protein_disease_scores, edge_weight_dict):
        """
        Forward pass with patient-aware edge coefficients.
        """
        # ========== 1. GNN Feature Learning ==========
        x_dict = {node_type: emb.weight for node_type, emb in self.embeddings.items()}
        
        h_dict = self.conv1(x_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_dict = {key: F.dropout(v, p=self.dropout_rate, training=self.training) 
                  for key, v in h_dict.items()}
        
        h_dict = self.conv2(h_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        
        h_patient = h_dict['Patient']
        h_protein = h_dict['Protein']
        
        # ========== 2. Patient-Aware Edge Coefficients ==========
        pat_idx, prot_idx = edge_index_dict[self.patient_protein_edge_type]
        
        # Extract edge features
        protein_relevance = protein_disease_scores[prot_idx]
        expression_values = edge_weight_dict[self.patient_protein_edge_type]
        
        # NEW: Get patient embeddings for each edge
        patient_embeddings = h_patient[pat_idx]  # [num_edges, hidden_channels]
        
        # Compute patient-aware edge coefficients
        edge_coeffs, rule_activations, patient_weights = self.edge_fuzzy_rules(
            protein_relevance,
            expression_values,
            patient_embeddings  # <- NEW: patient context
        )
        
        # ========== 3. Selective Message Passing ==========
        protein_messages = h_protein[prot_idx]
        scaled_messages = protein_messages * edge_coeffs.unsqueeze(-1)
        
        selective_aggregation = scatter_sum(
            scaled_messages, 
            pat_idx, 
            dim=0, 
            dim_size=h_patient.size(0)
        )
        
        # ========== 4. Classification ==========
        combined = torch.cat([h_patient, selective_aggregation], dim=-1)
        logits = self.classifier(combined)
        log_probs = F.log_softmax(logits, dim=1)
        
        return log_probs, edge_coeffs, rule_activations, patient_weights
    
    def get_learned_rules(self):
        """Extract learned fuzzy rule parameters."""
        return {
            'protein_relevance_threshold': self.edge_fuzzy_rules.theta[0].item(),
            'expression_threshold': self.edge_fuzzy_rules.theta[1].item(),
            'protein_relevance_sharpness': self.edge_fuzzy_rules.alpha[0].item(),
            'expression_sharpness': self.edge_fuzzy_rules.alpha[1].item(),
        }

In [ ]:
class PerEdgeFuzzyRuleLayer(nn.Module):
    """
    Per-edge learnable coefficients modulated by fuzzy rules.
    Each edge gets its own unique parameter that is scaled by rule activations.
    """
    def __init__(self, num_edges, num_features=2):
        super().__init__()
        
        # Per-edge learnable base coefficients (one parameter per edge)
        # Initialized close to 0.5 (neutral)
        self.edge_base_coeffs = nn.Parameter(
            torch.ones(num_edges) * 0.5
        )
        
        # Shared fuzzy rule parameters (thresholds and sharpness)
        self.theta = nn.Parameter(torch.randn(num_features))
        self.alpha = nn.Parameter(torch.ones(num_features))
        
        # Learnable weights to combine rule activations
        self.rule_weights = nn.Parameter(torch.ones(num_features))
    
    def forward(self, edge_indices, protein_relevance, expression_value):
        """
        Args:
            edge_indices: [num_batch_edges] indices into edge_base_coeffs
            protein_relevance: [num_batch_edges] disease scores
            expression_value: [num_batch_edges] expression values
            
        Returns:
            edge_coeffs: [num_batch_edges] final edge coefficients
            rule_activations: [num_batch_edges, 2] fuzzy rule activations
        """
        # Get base coefficients for these edges
        base_coeffs = self.edge_base_coeffs[edge_indices]  # [num_batch_edges]
        
        # Stack edge features
        edge_features = torch.stack([protein_relevance, expression_value], dim=-1)  # [num_batch_edges, 2]
        
        # Fuzzy rule activation: r_i = sigmoid(alpha_i * (feature_i - theta_i))
        rule_activations = torch.sigmoid(
            self.alpha * (edge_features - self.theta)
        )  # [num_batch_edges, 2]
        
        # Weighted combination of rules
        rule_score = (rule_activations * self.rule_weights).sum(dim=-1)  # [num_batch_edges]
        
        # Modulate base coefficient by rule score
        # edge_coeff = base_coeff * sigmoid(rule_score)
        edge_coeffs = base_coeffs * torch.sigmoid(rule_score)
        
        # Ensure final coefficient is in [0, 1]
        edge_coeffs = torch.sigmoid(edge_coeffs)
        
        return edge_coeffs, rule_activations


class HeteroFireGNN_PerEdge(nn.Module):
    """
    FireGNN with per-edge learnable parameters.
    Each patient-protein edge has its own unique learnable coefficient.
    """
    def __init__(self, data, hidden_channels, out_channels, heads=2, dropout_rate=0.5):
        super().__init__()
        
        self.dropout_rate = dropout_rate
        self.hidden_channels = hidden_channels
        self.patient_protein_edge_type = ('Patient', 'express', 'Protein')
        
        # Get number of patient-protein edges
        self.num_patient_protein_edges = data[self.patient_protein_edge_type].edge_index.size(1)
        
        # Node embeddings
        self.embeddings = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in {nt: data[nt].num_nodes for nt in data.node_types}.items()
        })
        
        # GNN layers for feature learning
        self.conv1 = HeteroConv({
            edge_type: GATConv((-1, -1), hidden_channels, heads, add_self_loops=False)
            for edge_type in data.edge_types
        })
        
        self.conv2 = HeteroConv({
            edge_type: GATConv(hidden_channels * heads, hidden_channels, heads=1, add_self_loops=False)
            for edge_type in data.edge_types
        })
        
        # Per-edge fuzzy rule layer
        self.edge_fuzzy_rules = PerEdgeFuzzyRuleLayer(
            num_edges=self.num_patient_protein_edges,
            num_features=2
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_channels, out_channels)
        )
    
    def forward(self, edge_index_dict, protein_disease_scores, edge_weight_dict, 
                patient_protein_edge_mapping=None):
        """
        Forward pass with per-edge learnable coefficients.
        
        Args:
            edge_index_dict: Dictionary of edge indices for all edge types
            protein_disease_scores: [num_proteins] disease-relation scores
            edge_weight_dict: Dictionary containing expression values
            patient_protein_edge_mapping: [num_patient_protein_edges] optional mapping 
                                         from current batch to global edge indices.
                                         If None, assumes all edges are present.
        """
        # ========== 1. GNN Feature Learning ==========
        x_dict = {node_type: emb.weight for node_type, emb in self.embeddings.items()}
        
        h_dict = self.conv1(x_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        h_dict = {key: F.dropout(v, p=self.dropout_rate, training=self.training) 
                  for key, v in h_dict.items()}
        
        h_dict = self.conv2(h_dict, edge_index_dict)
        h_dict = {key: F.elu(v) for key, v in h_dict.items()}
        
        h_patient = h_dict['Patient']
        h_protein = h_dict['Protein']
        
        # ========== 2. Per-Edge Coefficient Learning ==========
        pat_idx, prot_idx = edge_index_dict[self.patient_protein_edge_type]
        
        # Extract edge features
        protein_relevance = protein_disease_scores[prot_idx]
        expression_values = edge_weight_dict[self.patient_protein_edge_type]
        
        # Create edge indices (if batching, use provided mapping; else use range)
        if patient_protein_edge_mapping is None:
            edge_indices = torch.arange(
                len(pat_idx), 
                device=pat_idx.device
            )
        else:
            edge_indices = patient_protein_edge_mapping
        
        # Compute per-edge coefficients
        edge_coeffs, rule_activations = self.edge_fuzzy_rules(
            edge_indices,
            protein_relevance,
            expression_values
        )
        
        # ========== 3. Selective Message Passing ==========
        protein_messages = h_protein[prot_idx]
        scaled_messages = protein_messages * edge_coeffs.unsqueeze(-1)
        
        selective_aggregation = scatter_sum(
            scaled_messages, 
            pat_idx, 
            dim=0, 
            dim_size=h_patient.size(0)
        )
        
        # ========== 4. Classification ==========
        combined = torch.cat([h_patient, selective_aggregation], dim=-1)
        logits = self.classifier(combined)
        log_probs = F.log_softmax(logits, dim=1)
        
        return log_probs, edge_coeffs, rule_activations
    
    def get_learned_rules(self):
        """Extract learned fuzzy rule parameters."""
        return {
            'protein_relevance_threshold': self.edge_fuzzy_rules.theta[0].item(),
            'expression_threshold': self.edge_fuzzy_rules.theta[1].item(),
            'protein_relevance_sharpness': self.edge_fuzzy_rules.alpha[0].item(),
            'expression_sharpness': self.edge_fuzzy_rules.alpha[1].item(),
            'protein_relevance_weight': self.edge_fuzzy_rules.rule_weights[0].item(),
            'expression_weight': self.edge_fuzzy_rules.rule_weights[1].item(),
        }
    
    def get_edge_coefficient_statistics(self):
        """
        Get statistics about learned per-edge coefficients.
        Useful for understanding which edges are considered important.
        """
        with torch.no_grad():
            base_coeffs = torch.sigmoid(self.edge_fuzzy_rules.edge_base_coeffs)
            
            return {
                'mean': base_coeffs.mean().item(),
                'std': base_coeffs.std().item(),
                'min': base_coeffs.min().item(),
                'max': base_coeffs.max().item(),
                'num_high_coeffs': (base_coeffs > 0.7).sum().item(),  # "Strong" edges
                'num_low_coeffs': (base_coeffs < 0.3).sum().item(),   # "Weak" edges
            }


# ========== Training Loop ==========
def train_epoch(model, data, optimizer, protein_disease_scores, edge_weight_dict, 
                patient_labels, train_mask):
    """
    Training function with classification loss + regularization.
    """
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    log_probs, edge_coeffs, rule_activations = model(
        data.edge_index_dict,
        protein_disease_scores,
        edge_weight_dict
    )
    
    # Classification loss
    loss_cls = F.nll_loss(log_probs[train_mask], patient_labels[train_mask])
    
    # Regularization: push edge_coeffs toward 0 or 1 (more extreme)
    loss_reg = torch.mean(edge_coeffs * (1 - edge_coeffs))
    
    # Optional: L2 regularization on base coefficients to prevent overfitting
    base_coeffs = model.edge_fuzzy_rules.edge_base_coeffs
    loss_l2 = 0.0001 * torch.mean(base_coeffs ** 2)
    
    # Total loss
    loss = loss_cls + 0.1 * loss_reg + loss_l2
    
    loss.backward()
    optimizer.step()
    
    return loss.item(), loss_cls.item(), loss_reg.item()